In [3]:
import yfinance as yf 
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
def check_type(data):
    for i in range(0,len(data)):
        if(data['prompt'].iloc[i]==1):
            return 1
        elif(data['prompt'].iloc[i]==-1):
            return 0
def returns(data):
    if(check_type(data)==1):
        returns_p=[]
        for i in range(0,len(data)):
            if(data['prompt'].iloc[i]==1):
                for j in range(i+1,len(data)):
                    if(data['prompt'].iloc[j]==-1):
                        returns_p.append((data['Close'].iloc[j]-data['Close'].iloc[i])*100/data['Close'].iloc[i])
                        i=j
                        break
        returns_p=np.array(returns_p)
        return returns_p
    elif(check_type(data)==0):
        returns_p=[]
        for i in range(0,len(data)):
            if(data['prompt'].iloc[i]==-1):
                for j in range(i+1,len(data)):
                    if(data['prompt'].iloc[j]==1):
                        returns_p.append((data['Close'].iloc[i]-data['Close'].iloc[j])*100/data['Close'].iloc[j])
                        i=j
                        break
        returns_p=np.array(returns_p)
        return returns_p
def no_trades(data):
    return(len(returns(data)))
    
def MaxDrawDown(data):
    
#     max_peak_till_now = data["Close"].cummax()
#     drawdown = (data["Close"] - max_peak_till_now)/max_peak_till_now
#     return drawdown.min()*100
    final=np.zeros(no_trades(data))
    flag=0
    if(check_type(data)==1):
       
        for i in range(0,len(data)):
            if(data['prompt'].iloc[i]==1):
                 for j in range(i+1,len(data)):
                         if(data['prompt'].iloc[j]==-1):
                                max_peak_till_now = data["Close"].iloc[i:j+1].cummax()
                                drawdown = (data["Close"].iloc[i:j+1] - max_peak_till_now)/max_peak_till_now
                                final[flag]=drawdown.min()*100
                                flag=flag+1
                                break
                                
    elif(check_type(data)==0):
        for i in range(0,len(data)):
            if(data['prompt'].iloc[i]==-1):
                 for j in range(i+1,len(data)):
                         if(data['prompt'].iloc[j]==1):
                                min_low_till_now = data["Close"].iloc[i:j+1].cummin()
                                drawdown = (min_low_till_now-data["Close"].iloc[i:j+1])/min_low_till_now
                                final[flag]=drawdown.min()*100
                                flag=flag+1
                                break
    return final.min()
                        
                
            
            

            
def SharpeRatio(data):
    
    rfr = 0
    return(252 * (returns(data).mean() - rfr )/(np.sqrt(252) * returns(data).std()))

def portfolio_value(data):
    capital=10000000
 
    if(check_type(data)==0):
        for i in range(0,len(data)):
            if(data['prompt'].iloc[i]==-1):
                for j in range(0,len(data)):
                    if(data['prompt'].iloc[j]==1):
                        capital=(capital//data['Close'].iloc[j])*(data['Close'].iloc[i]) +capital%(data['Close'].iloc[j])
                        break
                        
    elif(check_type(data)==1):
           for i in range(0,len(data)):
            if(data['prompt'].iloc[i]==1):
                for j in range(0,len(data)):
                    if(data['prompt'].iloc[j]==-1):
                        capital=(capital//data['Close'].iloc[i])*(data['Close'].iloc[j]) +capital%(data['Close'].iloc[i])
                        break
        
    return capital
                        
def heiken_asi(data):
    hclose=np.zeros(len(data['Open']))
    hopen=np.zeros(len(data['Open']))
    hhigh=np.zeros(len(data['Open']))
    hlow=np.zeros(len(data['Open']))
    for i in range(1,len(data['Open'])):
        hclose[i]=(data['Open'].iloc[i]+data['High'].iloc[i]+data['Close'].iloc[i]+data['Low'].iloc[i])/4
        hopen[i]=(data['Open'].iloc[i-1]+data['Close'].iloc[i-1])/2
        np1=np.array([data['Open'].iloc[i],data['Close'].iloc[i],data['High'].iloc[i]])
        np2=np.array([data['Open'].iloc[i],data['Close'].iloc[i],data['Low'].iloc[i]])
        hhigh[i]=np.sort(np1)[-1]
        hlow[i]=np.sort(np2)[-1]
        
    df=pd.DataFrame()
    df['Close']=hclose
    df['Open']=hopen
    df['High']=hhigh
    df['Low']=hlow
    return(df)       
df=yf.download('^NSEI',start='2018-01-01',end='2024-01-01')
data=heiken_asi(df)
data['macd']=data['Close'].ewm(span=12,adjust="False").mean()-data['Close'].ewm(span=26,adjust='False').mean()
data['signal']=data['macd'].ewm(span=9,adjust='False').mean()
data['prompt']=np.zeros(len(data['Close']))
for i in range(0,len(data['Close'])-1):
    if(data['macd'].iloc[i]-data['signal'].iloc[i]<0 and data['macd'].iloc[i+1]-data['signal'].iloc[i+1]>=0):
        data['prompt'].iloc[i]=1
    elif(data['macd'].iloc[i]-data['signal'].iloc[i]>0 and data['macd'].iloc[i+1]-data['signal'].iloc[i+1]<=0):  
        data['prompt'].iloc[i]=-1
        
else:
    data['prompt'].iloc[i]=0

if(check_type(data)==1):
    data['stop_loss']=(1-0.05)*(data['Close'])
    for i in range(0,len(data)):
        if(data['prompt'].iloc[i]==1):
            for j in range(i+1,len(data)):
                if(data['Close'].iloc[j]<=data['stop_loss'].iloc[i]):
                    data['prompt'].iloc[j]=-1
                    i=j
                    break
            
elif(check_type(data)==0):
    data['stop_loss']=(1+0.05)*(data['Close'])
    for i in range(0,len(data)):
        if(data['prompt'].iloc[i]==-1):
            for j in range(i+1,len(data)):
                if(data['Close'].iloc[j]>=data['stop_loss'].iloc[i]):
                    data['prompt'].iloc[j]=1
                    i=j
                    break
for i in range(0,len(data)):
    if(data['prompt'].iloc[i]==1):
        for j in range(i+1,len(data)):
            if(data['prompt'].iloc[j]==-1):
                i=j
                break
            elif(data['prompt'].iloc[j]==1):
                data['prompt'].iloc[j]=0
for i in range(0,len(data)):
    if(data['prompt'].iloc[i]==-1):
        for j in range(i+1,len(data)):
            if(data['prompt'].iloc[j]==-1):
                data['prompt'].iloc[j]=0
            elif(data['prompt'].iloc[j]==1):
                i=j
                break



print(data.to_string())
flag=0
for i in range(0,len(data)):
    if(data['prompt'].iloc[i]==1):
        flag=flag+1
    elif(data["prompt"].iloc[i]==-1):
        flag=flag-1
# print(flag)
if(flag==1):
    data['prompt'].iloc[-1]=-1
elif(flag==-1):
    data['prompt'].iloc[-1]=1

print('THE NUMBER OF TRADES TAKEN IS:',no_trades(data))
print('RETURNS ON EVERY TRADE IS:',returns(data))
print('THE PORTFOLIO VALUE IS:',portfolio_value(data))
print('THE MAX DRAWDOWN IS:',MaxDrawDown(data),'%')
print('THE SHARPE RATIO IS:',SharpeRatio(data))

[*********************100%%**********************]  1 of 1 completed


             Close          Open          High           Low         macd      signal  prompt     stop_loss
0         0.000000      0.000000      0.000000      0.000000     0.000000    0.000000     0.0      0.000000
1     10464.750000  10459.875000  10503.599609  10482.650391   234.786058  130.436699     0.0  10987.987500
2     10482.162598  10462.925293  10513.000000  10504.799805   299.704003  199.808545     0.0  11006.270728
3     10544.824707  10487.100098  10566.099609  10558.849609   324.147365  241.928741     0.0  11072.065942
4     10608.762451  10546.549805  10631.200195  10623.599609   333.149634  269.064894     0.0  11139.200574
5     10636.212402  10607.649902  10659.150391  10645.099609   333.472889  286.523042     0.0  11168.023022
6     10633.112549  10641.049805  10655.500000  10652.049805   327.795812  296.968079     0.0  11164.768176
7     10641.299805  10642.125000  10664.599609  10651.200195   319.108022  302.288724     0.0  11173.364795
8     10662.824951  10644.12

THE MAX DRAWDOWN IS: -12.737441263439125 %
THE SHARPE RATIO IS: 4.022892927125841
